# Week 3 — Model Improvement: adding TPSA

First Week 3 experiment: add TPSA (topological polar surface area) back into the descriptor set as a 6th feature. Week 2's `featurize()` deliberately left it out (the plan's literal Week 2 list was MW/LogP/HBD/HBA/rotatable bonds), but TPSA was the single cleanest signal found anywhere in Week 1 EDA — near-clean class separation for BBB (non-permeable ~110, permeable ~40-50).

This notebook copies the reusable functions from `Featurization_Baseline.ipynb` (kept sealed/unedited as the Week 2 baseline reference) rather than editing that notebook in place, so Week 2's numbers stay a stable "before" to compare against. Only `featurize()` changes — everything else is identical to Week 2.

Each property section re-runs the same scaffold-split → featurize → train → evaluate pipeline, then compares directly against Week 2's frozen baseline numbers.

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.DataStructs import ConvertToNumpyArray


# Unchanged from Week 2: strips counter-ions/solvents from a salt, keeping
# only the largest fragment.
def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) == 1:
        return mol
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())


# Unchanged from Week 2: 2048-bit Morgan fingerprint, radius 2.
def compute_morgan_fp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr

In [2]:
# Week 3 change: adds TPSA as a 6th descriptor (2054 feature columns total,
# up from Week 2's 2053) — everything else identical to Week 2's featurize().
def featurize(df, smiles_col='Drug', radius=2, n_bits=2048):
    fp_rows = []
    desc_rows = []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(smi)
        mol = largest_fragment(mol)
        fp_rows.append(compute_morgan_fp(mol, radius=radius, n_bits=n_bits))
        desc_rows.append({
            'MolWt': Descriptors.MolWt(mol),
            'MolLogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        })

    fp_cols = [f'fp_{i}' for i in range(n_bits)]
    fp_df = pd.DataFrame(np.vstack(fp_rows), columns=fp_cols, index=df.index)
    desc_df = pd.DataFrame(desc_rows, index=df.index)
    return pd.concat([fp_df, desc_df], axis=1)

In [3]:
# Unchanged from Week 2: drops SMILES with conflicting labels, dedupes exact
# repeats. Used for classification targets (BBB, hERG, CYP3A4).
def dedupe_labels(df, smiles_col='Drug', y_col='Y'):
    label_counts = df.groupby(smiles_col)[y_col].nunique()
    conflicting = label_counts[label_counts > 1].index
    clean = df[~df[smiles_col].isin(conflicting)]
    return clean.drop_duplicates(subset=smiles_col)


# Unchanged from Week 2: averages Y across duplicate SMILES instead of
# dropping conflicts, appropriate for a continuous target. Used for Clearance.
def average_duplicate_targets(df, smiles_col='Drug', y_col='Y'):
    other_cols = [c for c in df.columns if c not in (smiles_col, y_col)]
    agg = {c: 'first' for c in other_cols}
    agg[y_col] = 'mean'
    return df.groupby(smiles_col, as_index=False).agg(agg)


# Unchanged from Week 2: scaffold split + a swappable dedup_fn.
def load_scaffold_split(dataset_class, name, dedup_fn=dedupe_labels):
    data = dataset_class(name=name)
    split = data.get_split(method='scaffold')
    train_df = dedup_fn(split['train'])
    test_df = dedup_fn(split['test'])
    return train_df, test_df

In [4]:
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, balanced_accuracy_score,
)


# Unchanged from Week 2.
def evaluate_regression(model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"RMSE: {rmse:.3f}")
    print(f"MAE:  {mae:.3f}")
    print(f"R^2:  {r2:.3f}")
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


# Unchanged from Week 2.
def evaluate_classification(model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    print(f"ROC-AUC:           {roc_auc:.3f}")
    print(f"F1:                {f1:.3f}")
    print(f"Balanced accuracy: {bal_acc:.3f}")
    return {'roc_auc': roc_auc, 'f1': f1, 'balanced_accuracy': bal_acc}

In [5]:
# Week 2's frozen baseline numbers (from Featurization_Baseline.ipynb /
# CLAUDE.md), used purely as a fixed reference to compare each Week 3 result
# against — not recomputed here, so Week 2 stays the stable "before" snapshot.
WEEK2_BASELINE = {
    'Solubility': {'rmse': 1.286, 'mae': 0.926, 'r2': 0.686},
    'BBB': {'roc_auc': 0.917, 'f1': 0.934, 'balanced_accuracy': 0.767},
    'hERG': {'roc_auc': 0.851, 'f1': 0.910, 'balanced_accuracy': 0.739},
    'CYP3A4': {'roc_auc': 0.879, 'f1': 0.750, 'balanced_accuracy': 0.779, 'auprc': 0.852},
    'Clearance': {'rmse': 44.480, 'mae': 35.069, 'r2': 0.115, 'spearman': 0.362},
}


# Prints each Week 2 metric next to the same metric computed this session
# (Week 3, +TPSA), with the change — the point of keeping Week 2 untouched.
def compare_to_week2(name, week3_metrics):
    print(f"\n--- {name}: Week 2 baseline -> Week 3 (+TPSA) ---")
    for metric, v2 in WEEK2_BASELINE[name].items():
        v3 = week3_metrics.get(metric)
        if v3 is None:
            continue
        print(f"{metric:>18}: {v2:.3f} -> {v3:.3f}  ({v3 - v2:+.3f})")

## Solubility (+TPSA)

In [6]:
from tdc.single_pred import ADME
from sklearn.ensemble import RandomForestRegressor

train_df, test_df = load_scaffold_split(ADME, 'Solubility_AqSolDB')
X_train = featurize(train_df)
X_test = featurize(test_df)
y_train = train_df['Y'].values
y_test = test_df['Y'].values

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

metrics = evaluate_regression(rf, X_test, y_test)
compare_to_week2('Solubility', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/9982 [00:00<?, ?it/s]

 12%|█▏        | 1233/9982 [00:00<00:00, 11824.26it/s]

...

 65%|██████▌   | 6511/9982 [00:00<00:00, 12659.76it/s]

 80%|███████▉  | 7969/9982 [00:00<00:00, 13281.16it/s]

 93%|█████████▎| 9324/9982 [00:00<00:00, 12875.40it/s]

100%|██████████| 9982/9982 [00:00<00:00, 12024.43it/s]

RMSE: 1.280
MAE:  0.920
R^2:  0.689

--- Solubility: Week 2 baseline -> Week 3 (+TPSA) ---
              rmse: 1.286 -> 1.280  (-0.006)
               mae: 0.926 -> 0.920  (-0.006)
                r2: 0.686 -> 0.689  (+0.003)


## BBB Permeability (+TPSA)

Expected biggest impact of the 5 — TPSA was the single cleanest class-separator found anywhere in Week 1 EDA for this exact property.

In [7]:
from sklearn.ensemble import RandomForestClassifier

bbb_train_df, bbb_test_df = load_scaffold_split(ADME, 'BBB_Martins')
X_bbb_train = featurize(bbb_train_df)
X_bbb_test = featurize(bbb_test_df)
y_bbb_train = bbb_train_df['Y'].values
y_bbb_test = bbb_test_df['Y'].values

rf_bbb = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_bbb.fit(X_bbb_train, y_bbb_train)

metrics = evaluate_classification(rf_bbb, X_bbb_test, y_bbb_test)
compare_to_week2('BBB', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/2030 [00:00<?, ?it/s]

 39%|███▉      | 801/2030 [00:00<00:00, 8006.55it/s]

 79%|███████▉  | 1602/2030 [00:00<00:00, 7693.57it/s]

100%|██████████| 2030/2030 [00:00<00:00, 7952.70it/s]

ROC-AUC:           0.910
F1:                0.936
Balanced accuracy: 0.779

--- BBB: Week 2 baseline -> Week 3 (+TPSA) ---
           roc_auc: 0.917 -> 0.910  (-0.007)
                f1: 0.934 -> 0.936  (+0.002)
 balanced_accuracy: 0.767 -> 0.779  (+0.012)


## hERG Blockade (+TPSA)

In [8]:
from tdc.single_pred import Tox

herg_train_df, herg_test_df = load_scaffold_split(Tox, 'hERG')
X_herg_train = featurize(herg_train_df)
X_herg_test = featurize(herg_test_df)
y_herg_train = herg_train_df['Y'].values
y_herg_test = herg_test_df['Y'].values

rf_herg = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_herg.fit(X_herg_train, y_herg_train)

metrics = evaluate_classification(rf_herg, X_herg_test, y_herg_test)
compare_to_week2('hERG', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/655 [00:00<?, ?it/s]

100%|██████████| 655/655 [00:00<00:00, 7328.75it/s]

ROC-AUC:           0.844
F1:                0.897
Balanced accuracy: 0.692

--- hERG: Week 2 baseline -> Week 3 (+TPSA) ---
           roc_auc: 0.851 -> 0.844  (-0.007)
                f1: 0.910 -> 0.897  (-0.013)
 balanced_accuracy: 0.739 -> 0.692  (-0.047)


## CYP3A4 Inhibition (+TPSA)

Week 1 EDA found essentially no separation from TPSA for this property specifically — a useful contrast case against BBB's expected-strong effect.

In [9]:
from sklearn.metrics import average_precision_score

cyp3a4_train_df, cyp3a4_test_df = load_scaffold_split(ADME, 'CYP3A4_Veith')
X_cyp3a4_train = featurize(cyp3a4_train_df)
X_cyp3a4_test = featurize(cyp3a4_test_df)
y_cyp3a4_train = cyp3a4_train_df['Y'].values
y_cyp3a4_test = cyp3a4_test_df['Y'].values

rf_cyp3a4 = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_cyp3a4.fit(X_cyp3a4_train, y_cyp3a4_train)

metrics = evaluate_classification(rf_cyp3a4, X_cyp3a4_test, y_cyp3a4_test)
y_cyp3a4_proba = rf_cyp3a4.predict_proba(X_cyp3a4_test)[:, 1]
metrics['auprc'] = average_precision_score(y_cyp3a4_test, y_cyp3a4_proba)
print(f"AUPRC:             {metrics['auprc']:.3f}")
compare_to_week2('CYP3A4', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/12328 [00:00<?, ?it/s]

  6%|▌         | 768/12328 [00:00<00:01, 7674.10it/s]

...

 87%|████████▋ | 10682/12328 [00:01<00:00, 7766.89it/s]

 93%|█████████▎| 11464/12328 [00:01<00:00, 7758.12it/s]

 99%|█████████▉| 12243/12328 [00:01<00:00, 7342.51it/s]

100%|██████████| 12328/12328 [00:01<00:00, 7775.25it/s]

ROC-AUC:           0.878
F1:                0.749
Balanced accuracy: 0.779
AUPRC:             0.851

--- CYP3A4: Week 2 baseline -> Week 3 (+TPSA) ---
           roc_auc: 0.879 -> 0.878  (-0.001)
                f1: 0.750 -> 0.749  (-0.001)
 balanced_accuracy: 0.779 -> 0.779  (-0.000)
             auprc: 0.852 -> 0.851  (-0.001)


## Clearance (+TPSA)

Uses `average_duplicate_targets` for the same reason as Week 2 (continuous target, real duplicate measurements — see NOTES.md). The assay-censoring issue (Y pinned at 3.0/150.0) is still not addressed here, unchanged from Week 2.

In [10]:
from scipy.stats import spearmanr

clearance_train_df, clearance_test_df = load_scaffold_split(
    ADME, 'Clearance_Hepatocyte_AZ', dedup_fn=average_duplicate_targets
)
X_clearance_train = featurize(clearance_train_df)
X_clearance_test = featurize(clearance_test_df)
y_clearance_train = clearance_train_df['Y'].values
y_clearance_test = clearance_test_df['Y'].values

rf_clearance = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_clearance.fit(X_clearance_train, y_clearance_train)

metrics = evaluate_regression(rf_clearance, X_clearance_test, y_clearance_test)
y_clearance_pred = rf_clearance.predict(X_clearance_test)
metrics['spearman'], _ = spearmanr(y_clearance_test, y_clearance_pred)
print(f"Spearman:          {metrics['spearman']:.3f}")
compare_to_week2('Clearance', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/1213 [00:00<?, ?it/s]

 54%|█████▍    | 654/1213 [00:00<00:00, 6527.77it/s]

100%|██████████| 1213/1213 [00:00<00:00, 6602.50it/s]

RMSE: 44.119
MAE:  34.950
R^2:  0.129
Spearman:          0.391

--- Clearance: Week 2 baseline -> Week 3 (+TPSA) ---
              rmse: 44.480 -> 44.119  (-0.361)
               mae: 35.069 -> 34.950  (-0.119)
                r2: 0.115 -> 0.129  (+0.014)
          spearman: 0.362 -> 0.391  (+0.029)


## Digging deeper: where does TPSA actually rank, and is hERG's drop real or noise?

Two concrete checks rather than just theorizing about the results above:

1. **TPSA's feature-importance rank** in each already-trained model, out of all 2054 features — tests the "TPSA may be redundant with fingerprint bits" hypothesis directly. If it ranks low despite being a strong *univariate* EDA signal, that supports redundancy with the fingerprint.
2. **hERG specifically:** since its test set is only 125 molecules, retrain the identical model with TPSA dropped (reusing the already-computed features, not recomputing anything) and check how many individual predictions actually flip, and whether those flips were close calls (probability near 0.5) or confident predictions changing — a few close calls flipping would support "small-test-set noise," not a real regression.

In [11]:
# TPSA's importance rank in each fitted model (1 = most important of 2054).
fitted_models = {
    'Solubility': (rf, X_train),
    'BBB': (rf_bbb, X_bbb_train),
    'hERG': (rf_herg, X_herg_train),
    'CYP3A4': (rf_cyp3a4, X_cyp3a4_train),
    'Clearance': (rf_clearance, X_clearance_train),
}

for name, (model, X_tr) in fitted_models.items():
    importances = pd.Series(model.feature_importances_, index=X_tr.columns)
    ranked = importances.sort_values(ascending=False)
    tpsa_rank = ranked.index.get_loc('TPSA') + 1
    tpsa_importance = importances['TPSA']
    percentile = 100 * (1 - tpsa_rank / len(ranked))
    print(f"{name:>10}: TPSA importance={tpsa_importance:.5f}, rank {tpsa_rank}/{len(ranked)} (top {100-percentile:.1f}%)")

Solubility: TPSA importance=0.03507, rank 3/2054 (top 0.1%)
       BBB: TPSA importance=0.03721, rank 2/2054 (top 0.1%)
      hERG: TPSA importance=0.02463, rank 3/2054 (top 0.1%)
    CYP3A4: TPSA importance=0.01488, rank 5/2054 (top 0.2%)
 Clearance: TPSA importance=0.07305, rank 1/2054 (top 0.0%)


In [12]:
# hERG without TPSA (reusing the already-computed features, just dropping
# the one column) — identical config otherwise, so any prediction difference
# is attributable to that one column alone, not random-seed variation.
X_herg_train_notpsa = X_herg_train.drop(columns='TPSA')
X_herg_test_notpsa = X_herg_test.drop(columns='TPSA')

rf_herg_notpsa = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_herg_notpsa.fit(X_herg_train_notpsa, y_herg_train)

pred_with_tpsa = rf_herg.predict(X_herg_test)
pred_without_tpsa = rf_herg_notpsa.predict(X_herg_test_notpsa)
proba_with_tpsa = rf_herg.predict_proba(X_herg_test)[:, 1]
proba_without_tpsa = rf_herg_notpsa.predict_proba(X_herg_test_notpsa)[:, 1]

flipped = pred_with_tpsa != pred_without_tpsa
print(f"Test set size: {len(y_herg_test)}")
print(f"Predictions that flipped when TPSA was added: {flipped.sum()}")
print(f"Of those, probability (without TPSA) for each flipped case:")
print(np.round(proba_without_tpsa[flipped], 3))
print(f"Of those, probability (with TPSA) for each flipped case:")
print(np.round(proba_with_tpsa[flipped], 3))

Test set size: 125
Predictions that flipped when TPSA was added: 3
Of those, probability (without TPSA) for each flipped case:
[0.492 0.48  0.493]
Of those, probability (with TPSA) for each flipped case:
[0.505 0.54  0.547]


## Experiment 2: Gradient Boosting (XGBoost) vs. Random Forest

Random Forest builds many trees independently and averages them. Gradient boosting builds trees sequentially — each new tree specifically targets the *errors* (residuals) left by the trees before it, rather than being trained independently. XGBoost is the classical, well-documented implementation of this, and it's what many of the strongest classical-ML entries on the TDC leaderboards are built on.

Testing on 2 properties first (Solubility — regression, BBB — classification) rather than all 5, per the plan's own Week 3 framing: this is a comparison to learn from, not a full re-run of the panel.

Reuses the exact same TPSA-augmented features already computed earlier in this notebook (`X_train`/`X_test` for Solubility, `X_bbb_train`/`X_bbb_test` for BBB) — only the model changes, so this isolates "does the model family matter" as its own question, separate from Experiment 1's "does the feature set matter."

Compared against **two** references: Week 2's frozen RF baseline (no TPSA), and this notebook's own Experiment 1 RF+TPSA result — so it's possible to tell whether any improvement comes from XGBoost itself, or is just carried over from adding TPSA.

In [13]:
# This experiment's own RF+TPSA results (from earlier in this notebook),
# kept as a second reference point alongside WEEK2_BASELINE.
WEEK3_RF_TPSA = {
    'Solubility': {'rmse': 1.280, 'mae': 0.920, 'r2': 0.689},
    'BBB': {'roc_auc': 0.910, 'f1': 0.936, 'balanced_accuracy': 0.779},
}


# Generic version of compare_to_week2 — prints any named metrics next to a
# given reference dict, so the same helper works for either comparison.
def compare_metrics(label, reference, new_metrics):
    print(f"\n--- {label} ---")
    for metric, v_old in reference.items():
        v_new = new_metrics.get(metric)
        if v_new is None:
            continue
        print(f"{metric:>18}: {v_old:.3f} -> {v_new:.3f}  ({v_new - v_old:+.3f})")

### Solubility — XGBoost

In [14]:
from xgboost import XGBRegressor

# n_estimators matched to the RF baseline (200) for a like-for-like
# comparison; no tuning yet (that's a separate Week 3 step).
xgb_sol = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb_sol.fit(X_train, y_train)

xgb_sol_metrics = evaluate_regression(xgb_sol, X_test, y_test)
compare_metrics('Solubility: Week 2 RF (no TPSA) -> XGBoost (+TPSA)', WEEK2_BASELINE['Solubility'], xgb_sol_metrics)
compare_metrics('Solubility: Week 3 RF+TPSA -> XGBoost (+TPSA)', WEEK3_RF_TPSA['Solubility'], xgb_sol_metrics)

RMSE: 1.264
MAE:  0.932
R^2:  0.696

--- Solubility: Week 2 RF (no TPSA) -> XGBoost (+TPSA) ---
              rmse: 1.286 -> 1.264  (-0.022)
               mae: 0.926 -> 0.932  (+0.006)
                r2: 0.686 -> 0.696  (+0.010)

--- Solubility: Week 3 RF+TPSA -> XGBoost (+TPSA) ---
              rmse: 1.280 -> 1.264  (-0.016)
               mae: 0.920 -> 0.932  (+0.012)
                r2: 0.689 -> 0.696  (+0.007)


### BBB — XGBoost

In [15]:
from xgboost import XGBClassifier

# XGBoost doesn't have class_weight='balanced' like sklearn's RF — the
# equivalent is scale_pos_weight = (count of negative class) / (count of
# positive class), computed here rather than hardcoded so it matches
# whatever the actual train split's imbalance is.
scale_pos_weight = (y_bbb_train == 0).sum() / (y_bbb_train == 1).sum()

xgb_bbb = XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1
)
xgb_bbb.fit(X_bbb_train, y_bbb_train)

xgb_bbb_metrics = evaluate_classification(xgb_bbb, X_bbb_test, y_bbb_test)
compare_metrics('BBB: Week 2 RF (no TPSA) -> XGBoost (+TPSA)', WEEK2_BASELINE['BBB'], xgb_bbb_metrics)
compare_metrics('BBB: Week 3 RF+TPSA -> XGBoost (+TPSA)', WEEK3_RF_TPSA['BBB'], xgb_bbb_metrics)

ROC-AUC:           0.907
F1:                0.921
Balanced accuracy: 0.823

--- BBB: Week 2 RF (no TPSA) -> XGBoost (+TPSA) ---
           roc_auc: 0.917 -> 0.907  (-0.010)
                f1: 0.934 -> 0.921  (-0.013)
 balanced_accuracy: 0.767 -> 0.823  (+0.056)

--- BBB: Week 3 RF+TPSA -> XGBoost (+TPSA) ---
           roc_auc: 0.910 -> 0.907  (-0.003)
                f1: 0.936 -> 0.921  (-0.015)
 balanced_accuracy: 0.779 -> 0.823  (+0.044)
